<a href="https://colab.research.google.com/github/bsheese/387/blob/main/23_language/23_3_Generation_and_Limits/23_3_1_Text_Generation.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# Language: Part 6
## Generating Text

---

## What This Notebook Is About

DistilBERT reads a whole text and gives one answer about it, such as positive or negative. The models behind chatbots do something different: they write. They produce text one token at a time, and each token is picked from a probability distribution over every token in the vocabulary, given all the text so far.

This notebook opens that process up with **GPT-2** (OpenAI, 2019), in its smallest version, with 124 million weights. GPT-2 was trained on about 8 million web pages with a single task: predict the next token. It is much smaller and older than current chatbots, but it works the same way at the level this notebook looks at, and it is small enough to run anywhere.

**What you will learn:**
1. How a language model gives a probability for every possible next token
2. Why its training loss is 19_1_4's cross-entropy again, and what the loss says about a piece of text
3. **Greedy decoding**: always picking the most likely token, and why it gets stuck
4. **Sampling** and **temperature**: picking tokens at random, and controlling how random
5. **Top-k** sampling and Hugging Face's `generate`

GPT-2 runs on the CPU in this notebook, even when a GPU is available. It's fast enough there (the slowest cell takes about 30 seconds on Colab's CPU), and on the CPU the random samples come out the same as the saved ones, which the text describes. The notebook is saved with its outputs. If you change a prompt or a seed, your text will differ from the saved text, which is expected.

In [1]:
import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

device = 'cpu'            # see above: GPT-2 small is fast enough on a CPU, and the samples match the saved ones
tokenizer = AutoTokenizer.from_pretrained('gpt2')
model = AutoModelForCausalLM.from_pretrained('gpt2').to(device).eval()
print(f'GPT-2 small: {sum(p.numel() for p in model.parameters()):,} weights, {len(tokenizer):,} tokens in its vocabulary')

Loading weights:   0%|          | 0/148 [00:00<?, ?it/s]

GPT-2 small: 124,439,808 weights, 50,257 tokens in its vocabulary


---

## Section 1: A Probability for Every Next Token

GPT-2 is a transformer like DistilBERT (23_1_1), with one important difference: its attention uses a **causal mask**, so each token can only look at the tokens before it. That means that at every position in a text, GPT-2's output is a prediction of the *next* token, made without seeing it. The model's output for a text of $n$ tokens is $n$ rows of 50,257 scores, one score for every token in the vocabulary; softmax turns a row into probabilities (19_1_4).

To continue a text, we only need the last row: the prediction for the token after the end.

In [2]:
prompt = 'The best thing about a beagle is'
ids = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
print('tokens:', tokenizer.convert_ids_to_tokens(ids[0]))

with torch.no_grad():
    scores = model(ids).logits
print('scores:', tuple(scores.shape), '(1 text, one row per token, one score per vocabulary token)')

probs = scores[0, -1].softmax(dim=0)          # the last row: what comes after "is"
top = probs.topk(10)
print('\nmost likely next tokens:')
for p, i in zip(top.values, top.indices):
    print(f'   {tokenizer.decode(int(i))!r:12} {p:.3f}')
print(f'the top 10 together: {top.values.sum():.3f}')

tokens: ['The', 'Ġbest', 'Ġthing', 'Ġabout', 'Ġa', 'Ġbe', 'agle', 'Ġis']


scores: (1, 8, 50257) (1 text, one row per token, one score per vocabulary token)

most likely next tokens:
   ' that'      0.455
   ' it'        0.116
   ' the'       0.068
   ' you'       0.043
   ' its'       0.042
   ' how'       0.021
   ','          0.018
   ' to'        0.017
   ' not'       0.014
   ' they'      0.014
the top 10 together: 0.806


The prompt is 8 tokens; "beagle" is split into " be" and "agle", since it isn't common enough to be one token in GPT-2's vocabulary (23_0_1). The scores have shape (1, 8, 50257): a prediction at each of the 8 positions, over all 50,257 tokens.

After "The best thing about a beagle is", the most likely next token is " that", with a probability of 0.455, far ahead of " it" (0.116) and " the" (0.068). All of the top ten are short, common words that could start the rest of the sentence. Together they have a probability of 0.806, so the remaining 50,247 tokens share about a fifth of the probability between them. That long tail matters in Section 4.

---

## Section 2: The Training Loss, Again

GPT-2 was trained to make the real next token as probable as possible, at every position of every training text. The loss is the cross-entropy loss from 19_1_4: for each position, take the probability the model gave to the token that really came next, and the penalty is $-\log$ of that probability. The loss of a text is the average penalty.

The next cell computes it by hand for a short sentence, and checks it against the loss the model reports when it is given the text as its own labels (`labels=ids`), which is how it was trained.

In [3]:
def token_report(text):
    ids = tokenizer(text, return_tensors='pt')['input_ids'].to(device)
    with torch.no_grad():
        out = model(ids, labels=ids)
    probs = out.logits[0, :-1].softmax(dim=1)              # predictions made at positions 0 .. n-2
    actual = ids[0, 1:]                                     # the tokens that really came next
    p_actual = probs[torch.arange(len(actual)), actual]
    return ids, p_actual, out.loss.item()

ids, p_actual, loss = token_report('The cat sat on the mat and looked out of the window.')
for tok, p in zip(tokenizer.convert_ids_to_tokens(ids[0, 1:]), p_actual):
    print(f'   {tok:10} probability {p:.3f}   penalty {-np.log(p.item()):.2f}')
print(f'average penalty by hand: {-np.log(p_actual.numpy()).mean():.3f}   model\'s loss: {loss:.3f}')

   Ġcat       probability 0.000   penalty 9.20
   Ġsat       probability 0.001   penalty 7.26
   Ġon        probability 0.224   penalty 1.50
   Ġthe       probability 0.454   penalty 0.79
   Ġmat       probability 0.004   penalty 5.51
   Ġand       probability 0.150   penalty 1.90
   Ġlooked    probability 0.054   penalty 2.93
   Ġout       probability 0.028   penalty 3.58
   Ġof        probability 0.073   penalty 2.62
   Ġthe       probability 0.794   penalty 0.23
   Ġwindow    probability 0.653   penalty 0.43
   .          probability 0.360   penalty 1.02
average penalty by hand: 3.079   model's loss: 3.079


The first two words are the most surprising to the model. After just "The", the probability of " cat" was about 0.0001 (it rounds to 0.000), a penalty of 9.20, because almost any noun could come next. " sat" after "The cat" got 0.001. Once the sentence is under way, the model expects much of what comes: " the" after "on" (0.454), " the" after "out of" (0.794) and " window" after "looked out of the" (0.653). " mat" got only 0.004 (penalty 5.51), although "the cat sat on the mat" is a well-known phrase; the model apparently gave more weight to other things a cat might sit on.

The average of the twelve penalties is 3.079, exactly the loss the model reports. Training GPT-2 meant running this calculation on billions of tokens of web text and adjusting the weights to bring the average down.

The loss is also a way to ask how "expected" a piece of text is. Here is the same sentence with its words in reverse order.

In [4]:
for text in ['The cat sat on the mat and looked out of the window.',
             'Window the of out looked and mat the on sat cat the.']:
    _, _, loss = token_report(text)
    print(f'loss {loss:.3f}   e^loss {np.exp(loss):7.1f}   {text}')

loss 3.079   e^loss    21.7   The cat sat on the mat and looked out of the window.
loss 7.361   e^loss  1573.3   Window the of out looked and mat the on sat cat the.


The normal sentence has a loss of 3.079 and the reversed one 7.361. The second column, $e^{\text{loss}}$, is called the **perplexity**, and it's easier to picture: 21.7 means the model was, on average, about as uncertain as if it were choosing among 22 equally likely tokens at each step. For the reversed sentence it's 1,573. The model has learned what ordinary English looks like, and the loss measures how far a text is from that. 23_3_2 uses this to check whether a model has seen a particular text before.

---

## Section 3: Greedy Decoding

The simplest way to write text with this model is to take the single most likely next token, add it to the text, and repeat. That is called **greedy decoding**. The loop below does it for 40 tokens. Each pass runs the whole text through the model again; real systems save work by keeping the earlier positions' results, but the answer is the same.

In [5]:
ids = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
for _ in range(40):
    with torch.no_grad():
        next_id = model(ids).logits[0, -1].argmax()        # the single most likely next token
    ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
print(tokenizer.decode(ids[0]))

The best thing about a beagle is that it's a very good dog. It's a very good dog. It's a very good dog. It's a very good dog. It's a very good dog. It's a very


The start is reasonable: "that it's a very good dog." But then the model repeats "It's a very good dog." over and over. Once the sentence has appeared, it makes itself more likely: after one copy, the most likely continuation is another copy, and greedy decoding has no way to choose anything else. This repetition is a well-known problem with greedy decoding, and it's the main reason text generators don't use it on its own.

---

## Section 4: Sampling and Temperature

Instead of always taking the top token, we can **sample**: pick the next token at random, with each token's chance of being picked equal to its probability. `torch.multinomial` does that. A token with probability 0.455 is picked about 45% of the time, and a token with probability 0.001 about once in a thousand.

**Temperature** controls how random the picks are. Before softmax, every score is divided by the temperature $T$. 19_1_4 showed that multiplying scores by 3 made softmax's top probability go from 0.64 to 0.95; dividing by a temperature below 1 does the same thing, making the likely tokens even more likely. A temperature above 1 squeezes the scores together, so unlikely tokens get more of a chance. The next cell shows the effect on the first token after our prompt.

In [6]:
with torch.no_grad():
    last = model(tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)).logits[0, -1]
top_ids = last.topk(5).indices
print(f"{'temperature':12}" + ''.join(f'{tokenizer.decode(int(i))!r:>9}' for i in top_ids))
for T in [0.3, 0.7, 1.0, 1.5, 3.0]:
    p = (last / T).softmax(dim=0)
    print(f'{T:<12}' + ''.join(f'{p[i]:9.3f}' for i in top_ids))

temperature   ' that'    ' it'   ' the'   ' you'   ' its'
0.3             0.987    0.010    0.002    0.000    0.000
0.7             0.737    0.104    0.048    0.025    0.024
1.0             0.455    0.116    0.068    0.043    0.042
1.5             0.110    0.044    0.031    0.023    0.022
3.0             0.003    0.002    0.002    0.001    0.001


At temperature 1 the probabilities are the model's own: " that" at 0.455. At 0.3 the top token takes almost everything (0.987), so sampling would nearly always pick it, which is close to greedy decoding. At 1.5 the top five tokens have only 0.23 of the probability between them, and at 3.0 each of them is about as likely as any other, so sampling would pick almost anything from the vocabulary.

Now whole continuations, sampled at three temperatures. The random generator is seeded, so the cell gives the same text every time it runs on the same kind of computer.

In [7]:
def sample(prompt, n_tokens, temperature, seed):
    torch.manual_seed(seed)
    ids = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
    for _ in range(n_tokens):
        with torch.no_grad():
            probs = (model(ids).logits[0, -1] / temperature).softmax(dim=0)
        next_id = torch.multinomial(probs, 1)                 # pick at random, by probability
        ids = torch.cat([ids, next_id.view(1, 1)], dim=1)
    return tokenizer.decode(ids[0])

for T in [0.3, 1.0, 1.6]:
    print(f'temperature {T}:')
    print('   ', sample(prompt, 40, T, seed=0).replace('\n', ' '), '\n')

temperature 0.3:


    The best thing about a beagle is that it's a pretty good dog. It's a good dog. It's a good dog. It's a good dog. It's a good dog.  I'm not sure if it 

temperature 1.0:


    The best thing about a beagle is a bull pilot with pretty much every barrel on the plane, the only defect being a small, gas leak on the bottom of the can so a lot of shots can sail out the front. The single 

temperature 1.6:


    The best thing about a beagle is along gritty pilot highwayave stories... is "tree legs diffuse the spill that squat unfortunately CathP wrestler roamer regulatesPlus Chin decreases can soBO82 arm velor sail barrel doines spike Sanskrit sun 



The three temperatures behave as the table suggests. At 0.3 the text is close to the greedy text and gets stuck the same way: "It's a good dog." four times. At 1.0 it is fluent and grammatical but wanders off immediately, into a "bull pilot" and a gas leak on a plane, because every token is sampled from the full distribution, and over 40 tokens some unlikely ones get picked and take the text somewhere strange. At 1.6 it falls apart into fragments and made-up words ("highwayave", "CathP").

None of these is what we would want. Low temperatures repeat themselves, and high ones lose the thread. Section 5 adds a common fix.

---

## Section 5: Top-k Sampling and `generate`

High temperatures go wrong because, out of 50,257 tokens, a great many unlikely ones each get a small chance, and together they get picked often. **Top-k sampling** removes that tail: keep only the $k$ most likely tokens, set every other token's probability to 0, and sample from those $k$. A related method, **top-p** (or nucleus) sampling, keeps the smallest set of top tokens whose probabilities add up to $p$, for example 0.9, so the number of tokens kept changes with how sure the model is.

Hugging Face's `generate` method does all of this, with many more options, and it is what most code uses. The cell below asks it for three continuations at temperature 0.8 with top-k 50, settings close to common defaults.

In [8]:
torch.manual_seed(1)
ids = tokenizer(prompt, return_tensors='pt')['input_ids'].to(device)
outputs = model.generate(ids, max_new_tokens=40, do_sample=True, temperature=0.8, top_k=50,
                         num_return_sequences=3, pad_token_id=tokenizer.eos_token_id)
for out in outputs:
    print('-', tokenizer.decode(out, skip_special_tokens=True).replace('\n', ' '), '\n')

- The best thing about a beagle is that it's a big animal and so it can do something for you," she says.  But the long-distance flycatcher has a special trait that makes it a formidable threat. " 

- The best thing about a beagle is that there are no problems. The dogs I have are in good health and the owners are happy to provide you with a variety of dog training services, including:  Training exercises for dogs   

- The best thing about a beagle is that it gets to ride the dog's back! I'm trying to do that with my own dogs as well!"  - Tom  "It's been very fun getting to know the Be 



These three are much better than the samples in Section 4: each is a few sentences of fluent, varied text that sounds like something from the web, and none of them repeats itself or falls into nonsense. They are still not sensible as statements about beagles. The first says that a beagle is "a big animal" and then turns to a "long-distance flycatcher", and the third says a beagle "gets to ride the dog's back". GPT-2 produces text that has the form of its training text, a quotation from an interview, an ad for dog training, a forum post, with no idea of what's true about beagles.

---

## Summary

- A language model like GPT-2 gives, for every position in a text, a probability for each of the tokens in its vocabulary (50,257 for GPT-2) to come next. A **causal mask** stops it from seeing the tokens it is predicting.
- It is trained with the **cross-entropy loss** of 19_1_4: the average of $-\log$ of the probability it gave to each real next token. A low loss means the text was what the model expected. The normal sentence had a loss of 3.079 (perplexity 21.7), and the same words in reverse 7.361 (perplexity 1,573).
- **Greedy decoding** always takes the most likely token. It tends to get stuck repeating itself.
- **Sampling** picks tokens at random according to their probabilities. **Temperature** divides the scores before softmax: below 1 makes the text safer and more repetitive, above 1 makes it more varied and, past a point, nonsense.
- **Top-k** and **top-p** sampling cut off the long tail of unlikely tokens, which keeps sampled text more coherent. `generate` does all of this in one call.
- Nothing in any of these methods checks whether what is written is true. Every token is chosen because it is likely to come next.

## What's next

GPT-2 continues text; it doesn't answer questions or follow instructions. Chatbots are language models like it, much larger, that have also been fine-tuned on examples of instructions and good responses. 23_3_2 uses a small model of that kind to look at what such models get wrong: answers that are fluent and false, mistakes that come from tokens, answers that change when a question is reworded, and how to measure all of this fairly.